# Metro Ridership Scraper

**Status: Obsolete (scraper cells)**

This notebook originally scraped ridership data from the LA Metro website using Selenium.
**The scraper approach is no longer used** — ridership CSVs are now obtained directly from
LA Metro or via public records request.

Kept for historical reference. The helper functions (`str_to_float`, `clean_up_df`) may
still be useful for reference when exploring raw data formats.

**Do not run the Selenium/ChromeDriver cells** — they require a local ChromeDriver install
and hardcoded Windows paths that are no longer maintained.

# Scrape Metro Ridership website

Get all data from https://opa.metro.net/MetroRidership/

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
from datetime import datetime
import time
import os
import json

pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 1000)

In [ ]:
today = datetime.now().strftime('%Y_%m_%d')
url = 'https://opa.metro.net/MetroRidership/IndexBusDO.aspx'

In [ ]:
def str_to_float(col):
    col = pd.to_numeric(
        col.str.replace(',', ''),
        errors='coerce'
    ).astype(float)
    return col

def clean_up_df(df):
    print('formatting data')
    # make the table look like we store
    df[[
        'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
    ]] = df[[
        'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
    ]].apply(str_to_float, axis=1)
    
    df['year'] = df.day_type.str[:4]
    df['month'] = df.day_type.str[-2:]
    df = df[[
        'year', 'month', 'line_name',
        'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
    ]].sort_values(
        by=['year', 'month', 'line_name']
    ).reset_index(drop=True)
    
    return df

In [ ]:
# Set the path to your local web driver
driver_path = 'F:\progs\chromedriver\chromedriver.exe'

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument("--enable-logging")
driver = webdriver.Chrome(executable_path=driver_path, options=chrome_options)

In [ ]:
# set parameters
table_id = 'ContentPlaceHolder1_ASPxRoundPanel2_GridView1'
years = [str(y) for y in range(2009, 2025)]
periods = ['1'] # download a table of 13 months for every January

cols_dict = {
    'Period': 'day_type',
    'Estimated Weekday Ridership': 'est_wkday_ridership',
    'Estimated Saturday Ridership': 'est_sat_ridership',
    'Estimated Sunday Ridership': 'est_sun_ridership',
}
df_cols = ['line_name'] + list(cols_dict.values())

In [ ]:
driver.get(url)
# driver.find_element(By.ID, 'ContentPlaceHolder1_rpSys_btnSys').click()
select_line = Select(driver.find_element_by_id('ContentPlaceHolder1_lbLines'))
line_names = []
for line in select_line.options:
     line_names.append(line.text)

In [ ]:
for year in years:
    df_all = pd.DataFrame(columns = df_cols)
    for period in periods:
        print(year, period)
        driver.get(url)
        # select year and period
        select_year = Select(driver.find_element_by_id(
            'ContentPlaceHolder1_ddlYear'
        ))
        select_year.select_by_value(year)
        select_period = Select(driver.find_element_by_id(
            'ContentPlaceHolder1_ddlPeriod'
        ))
        select_period.select_by_value(period)
        
        for line_name in line_names:
            select_line = Select(
                driver.find_element_by_id(
                    'ContentPlaceHolder1_lbLines'
                )
            )
            select_line.select_by_value(line_name)
            driver.find_element(By.ID, 'ContentPlaceHolder1_btnSubmit').click()
            time.sleep(5)

            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            table = soup.find('table', id=table_id)
            if table:
                print('table:', line_name)
                rows = table.find_all('tr')
                data = []
                for row in rows:
                    cells = row.find_all(['th', 'td'])
                    cell_data = [cell.get_text(strip=True) for cell in cells]
                    data.append(cell_data)

                df = pd.DataFrame(data)
                #display(df.head(1))
                df.columns = df.iloc[0]
                df = df.drop(0).rename(
                    columns = cols_dict
                ).reset_index(drop=True)
                df['line_name'] = line_name
                df_all = pd.concat([df_all, df]).reset_index(drop=True)
            else:
                print(f'no table found for line: {line_name}')
    df_all.to_csv(f'downloads/metro_ridership_{year}_{today}.csv', index=False)

In [ ]:
driver.quit()

## brush up the output

In [ ]:
dfs_scraped = pd.DataFrame()
for y in years:
    df = pd.read_csv(f'downloads/metro_ridership_{y}_2024_05_10.csv')
    dfs_scraped = pd.concat([dfs_scraped, df]).reset_index(drop=True)

print(dfs_scraped.shape)
df_temp = dfs_scraped.drop_duplicates(keep='last').copy()
print(df_temp.shape)

dfs_scraped_clean = clean_up_df(
    df_temp
)

In [ ]:
dfs_scraped_clean.head()

In [ ]:
dfs_scraped_clean.year.unique()

In [ ]:
dfs_scraped_clean.to_csv(f'metro_ridership_scraped_{today}.csv', index=False)